# Agent nieliniowy (planer) — przebieg z human-in-the-loop

Ten notebook uruchamia **`agent_planner`** — planer („mózg operacji") sam decyduje,
co dalej (analizuj / zapytaj człowieka / pisz / brak podstaw), więc graf **zawraca**
w pętli. Gdy planer poprosi o decyzję, graf **pauzuje** i czeka, aż wpiszesz odpowiedź
(wcielasz się w radcę).

> Konfiguracja LLM (klucz, model) jest w pliku `.env` — wczytuje ją pierwsza komórka.

## Setup

Kompilujemy graf z checkpointerem (potrzebny do pauzy/wznawiania) i świeżym `thread_id`.

In [ ]:
import os

# Konfiguracja LLM (klucz, model) wczytana z pliku .env — nic na sztywno w komórce.
from dotenv import load_dotenv

load_dotenv()
print("model:", os.environ.get("LLM_MODEL"), "| klucz z .env:", bool(os.environ.get("LLM_API_KEY")))

import uuid
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from agent_planner.graph import build_graph

GOAL = (
    "Oceń, czy w sprawie zasadna jest apelacja od wyroku sądu pierwszej instancji, "
    "a jeśli tak — sporządź apelację z perspektywy obrony."
)

graph = build_graph(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": uuid.uuid4().hex}}  # uruchom tę komórkę ponownie, by zacząć od nowa
GOAL

## Diagram grafu

Nieliniowy — planer jest centrum, do którego wszystko wraca.

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("PNG wymaga internetu — diagram w wersji tekstowej (mermaid):\n")
    print(graph.get_graph().draw_mermaid())

## Uruchomienie z human-in-the-loop

Graf rusza i kręci się w pętli planera. Gdy planer wybierze `ask_human`, **zatrzyma
się** i wypisze pytanie — wpisz odpowiedź (np. `pisz` albo `analizuj dalej, sprawdź jeszcze X`)
i naciśnij Enter, a graf wznowi działanie.

In [ ]:
state = graph.invoke({"goal": GOAL}, config=config)

while "__interrupt__" in state:
    print("\n=== PLANER PYTA ===")
    print(state["__interrupt__"][0].value)
    answer = input("\nTwoja decyzja (radca): ")
    state = graph.invoke(Command(resume=answer), config=config)

print("\n=== GOTOWE ===")

## Wynik

In [ ]:
print(state.get("document", "(brak dokumentu)"))

In [ ]:
# Zapis apelacji do data/output/ (nazwa: podejście + znacznik czasu; poza gitem)
from src.output import save_appeal

document = state.get("document", "")
if document.strip():
    path = save_appeal(document, "agent_planner")
    print("Zapisano apelację do:", path)
else:
    print("Brak dokumentu do zapisania.")

## Podgląd przebiegu (opcjonalnie)

Ile rund wykonał planer i jakie analizy zebrał.

In [ ]:
print("Rundy planera:", state.get("rounds"))
print("Wykonane analizy:", len(state.get("task_outputs", [])))
for t in state.get("task_outputs", []):
    print("  •", t.action)

decision = state.get("decision")
if decision is not None:
    print("\nOstatnia decyzja planera:", decision.action)
    print(decision.reasoning)